In [ ]:
!pip install pennylane pennylane-lightning[gpu] numpy scipy custatevec_cu12
!pip install pennylane-qchem --no-deps
!pip install h5py --upgrade
!pip install openfermionpyscf

# Quantum Simulation of H₂, VQE, QFT, and Strict Single-Shot Contextual Correction

This notebook combines an H₂ molecular Hamiltonian, a compact VQE ground-state demonstration, explicit QFT evolution, contextual error injection inside circuit depth, strict single-shot measurement, and leakage-audited deterministic reference correction.

All legitimate molecular-state preparation and QFT evolution are propagated independently in the ideal trajectory. Only noisy-vs-ideal displacement at identical circuit context is classified as contextual error.

# Contextual Error Injection and Deterministic Reference Correction

This extension keeps the molecular Hamiltonian and QFT workflow, but separates three trajectories:

- **Ideal**: legitimate Hartree–Fock → QFT evolution.
- **Noisy**: the same QFT with gate/qubit-dependent coherent errors injected *inside* the QFT. An early error therefore propagates through all subsequent controlled-phase and SWAP operations.
- **Corrected**: deterministic contextual-reference projection using the ideal trajectory as the reference.

The QFT-induced change is **not** treated as an error. We explicitly audit the legitimate QFT displacement separately from the context-propagated error displacement. The molecular Hamiltonian expectation is evaluated for the pre-QFT molecular state and for the three post-QFT trajectories.


In [ ]:
import pennylane as qml
from pennylane import qchem
from pennylane import numpy as np
from pennylane.qchem import import_operator

SEED = 42
NOISE_SEED = SEED + 15015
RX_ERROR_RANGE = (-0.060, 0.060)
RY_ERROR_RANGE = (-0.080, 0.080)
RZ_ERROR_RANGE = (-0.060, 0.060)
EPS = 1e-12
SHOTS = 1
SHOT_SEED = SEED + 909
shot_rng = np.random.default_rng(SHOT_SEED)

# H2 / STO-3G
symbols = ["H", "H"]
coordinates = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.74])
hf_file = qchem.meanfield(symbols=symbols, coordinates=coordinates,
                          name="h2", charge=0, basis="sto-3g")
q_op = qchem.decompose(hf_file=hf_file, mapping="jordan_wigner")
H_mol = import_operator(q_op)
qubits = len(H_mol.wires)
wires = list(range(qubits))
hf_state = np.array([1, 1] + [0] * (qubits - 2), dtype=int)

try:
    dev = qml.device("lightning.gpu", wires=qubits)
    BACKEND = "lightning.gpu"
except Exception:
    dev = qml.device("lightning.qubit", wires=qubits)
    BACKEND = "lightning.qubit"

print("Backend             :", BACKEND)
print("Qubits              :", qubits)
print("Hartree-Fock state  :", hf_state)
print("Hamiltonian terms   :", len(H_mol.terms()[0]))
print("Strict measurement shots :", SHOTS)


In [ ]:
# Explicit QFT operation list: this makes circuit-depth error propagation auditable.
def build_qft_operations(n):
    ops = []
    for target in range(n):
        ops.append(("H", target, None, None))
        for control in range(target + 1, n):
            angle = float(np.pi / (2 ** (control - target)))
            ops.append(("CPHASE", target, control, angle))
    for i in range(n // 2):
        ops.append(("SWAP", i, n - i - 1, None))
    return ops

QFT_OPERATIONS = build_qft_operations(qubits)
noise_rng = np.random.default_rng(NOISE_SEED)
ERROR_MAP = np.zeros((len(QFT_OPERATIONS), qubits, 3), dtype=float)
ERROR_MAP[:, :, 0] = noise_rng.uniform(*RX_ERROR_RANGE, size=(len(QFT_OPERATIONS), qubits))
ERROR_MAP[:, :, 1] = noise_rng.uniform(*RY_ERROR_RANGE, size=(len(QFT_OPERATIONS), qubits))
ERROR_MAP[:, :, 2] = noise_rng.uniform(*RZ_ERROR_RANGE, size=(len(QFT_OPERATIONS), qubits))

def apply_qft_op(op):
    kind, a, b, angle = op
    if kind == "H": qml.Hadamard(a)
    elif kind == "CPHASE": qml.ControlledPhaseShift(angle, wires=[b, a])
    elif kind == "SWAP": qml.SWAP(wires=[a, b])

def inject_contextual_error(k):
    for q in wires:
        ex, ey, ez = ERROR_MAP[k, q]
        qml.RX(ex, q); qml.RY(ey, q); qml.RZ(ez, q)

print("Explicit QFT operations :", len(QFT_OPERATIONS))
print("Noise seed              :", NOISE_SEED)


In [ ]:
@qml.qnode(dev)
def pre_qft_state():
    qml.BasisState(hf_state, wires=wires)
    return qml.state()

@qml.qnode(dev)
def ideal_qft_state():
    qml.BasisState(hf_state, wires=wires)
    for op in QFT_OPERATIONS:
        apply_qft_op(op)
    return qml.state()

@qml.qnode(dev)
def noisy_qft_state():
    qml.BasisState(hf_state, wires=wires)
    for k, op in enumerate(QFT_OPERATIONS):
        apply_qft_op(op)
        inject_contextual_error(k)  # error propagates through every later QFT operation
    return qml.state()

psi_pre = np.asarray(pre_qft_state(), dtype=complex)
psi_ideal = np.asarray(ideal_qft_state(), dtype=complex)
psi_noisy = np.asarray(noisy_qft_state(), dtype=complex)

def normalize(v):
    return v / (np.linalg.norm(v) + EPS)

def align_phase(reference, state):
    z = np.vdot(reference, state)
    return state if abs(z) < EPS else state * np.conj(z / abs(z))

psi_pre = normalize(psi_pre)
psi_ideal = normalize(psi_ideal)
psi_noisy = align_phase(psi_ideal, normalize(psi_noisy))

# No correction is performed in the injection/trajectory cell.
# Correction is isolated below behind an explicit information boundary.
print("Independent ideal and noisy state trajectories generated.")


## Stage-by-Stage QFT Evolution and Contextual Error Propagation

The molecular input state is part of the legitimate computation. For the Hartree--Fock branch it is $|\psi_0\rangle=|\mathrm{HF}\rangle$; for the VQE branch it is the optimized variational molecular state.

At QFT depth $l$, the clean contextual trajectory is

$$
|\psi_l^{\mathrm{ideal}}\rangle
=
U_l|\psi_{l-1}^{\mathrm{ideal}}\rangle.
$$

The legitimate circuit displacement is

$$
D_l^{\mathrm{legit}}
=
|\psi_l^{\mathrm{ideal}}\rangle
-
|\psi_{l-1}^{\mathrm{ideal}}\rangle.
$$

The noisy trajectory receives the contextual error operator **after** the same legitimate gate:

$$
|\psi_l^{\mathrm{noisy}}\rangle
=
E_l U_l|\psi_{l-1}^{\mathrm{noisy}}\rangle.
$$

The same-depth error is

$$
D_l^{\mathrm{error}}
=
|\psi_l^{\mathrm{noisy}}\rangle
-
|\psi_l^{\mathrm{ideal}}\rangle.
$$

Because the noisy state from depth $l$ is used as the input at depth $l+1$, an early error is propagated by every later QFT operation. Comparing clean and noisy states at identical depth prevents legitimate molecular-state preparation and legitimate QFT evolution from being classified as error.

In [ ]:
# =====================================================================
# STAGE-BY-STAGE QFT PROPAGATION AUDIT — HARTREE-FOCK INPUT
# =====================================================================

def apply_operation_to_state(state, op):
    audit_dev = qml.device("default.qubit", wires=qubits)
    @qml.qnode(audit_dev)
    def circuit():
        qml.StatePrep(state, wires=wires)
        apply_qft_op(op)
        return qml.state()
    return normalize(np.asarray(circuit(), dtype=complex))

def apply_error_to_state(state, k):
    audit_dev = qml.device("default.qubit", wires=qubits)
    @qml.qnode(audit_dev)
    def circuit():
        qml.StatePrep(state, wires=wires)
        inject_contextual_error(k)
        return qml.state()
    return normalize(np.asarray(circuit(), dtype=complex))

ideal_stage = psi_pre.copy()
noisy_stage = psi_pre.copy()

print("=" * 112)
print("STAGE-BY-STAGE EXPECTED-TRAJECTORY / ERROR-PROPAGATION AUDIT")
print("=" * 112)
print(f"{'Stage':<18}{'Legitimate RMS':>20}{'Same-depth error RMS':>26}{'Ideal/noisy fidelity':>24}")
print("-" * 112)

for k, op in enumerate(QFT_OPERATIONS):
    previous_ideal = ideal_stage.copy()
    ideal_stage = apply_operation_to_state(ideal_stage, op)

    noisy_stage = apply_operation_to_state(noisy_stage, op)
    noisy_stage = apply_error_to_state(noisy_stage, k)
    noisy_stage = align_phase(ideal_stage, noisy_stage)

    legitimate_rms = float(np.sqrt(np.mean(np.abs(ideal_stage - previous_ideal) ** 2)))
    same_depth_error_rms = float(np.sqrt(np.mean(np.abs(noisy_stage - ideal_stage) ** 2)))
    fidelity = float(np.abs(np.vdot(ideal_stage, noisy_stage)) ** 2)

    print(
        f"{k+1}:{op[0]:<16}"
        f"{legitimate_rms:>20.8e}"
        f"{same_depth_error_rms:>26.8e}"
        f"{fidelity:>24.10f}"
    )

print("-" * 112)
print("Legitimate RMS is QFT evolution; same-depth error RMS is noisy-vs-ideal.")
print("Injected errors remain in the noisy state and propagate through all later QFT operations.")

In [ ]:
# Separate legitimate QFT displacement from context-propagated error displacement.
qft_displacement = psi_ideal - psi_pre
contextual_error_displacement = psi_noisy - psi_ideal
# AUDIT-ONLY exact-state reference projection.
# This is kept separate from the strict single-shot measurement correction below.
state_error_residual_audit = psi_noisy - psi_ideal
psi_corrected_audit = normalize(psi_noisy - state_error_residual_audit)
post_correction_residual = psi_corrected_audit - psi_ideal

qft_rms = float(np.sqrt(np.mean(np.abs(qft_displacement) ** 2)))
error_rms = float(np.sqrt(np.mean(np.abs(contextual_error_displacement) ** 2)))
corrected_rms = float(np.sqrt(np.mean(np.abs(post_correction_residual) ** 2)))

qft_norm = float(np.linalg.norm(qft_displacement))
err_norm = float(np.linalg.norm(contextual_error_displacement))
alignment = float(np.real(np.vdot(qft_displacement, contextual_error_displacement)) /
                  (qft_norm * err_norm + EPS))

print("=" * 82)
print("QFT / CONTEXTUAL ERROR DECOMPOSITION")
print("=" * 82)
print(f"Legitimate QFT displacement RMS : {qft_rms:.8e}")
print(f"Context-propagated error RMS    : {error_rms:.8e}")
print(f"Post-correction residual RMS    : {corrected_rms:.8e}")
print(f"QFT↔error real alignment        : {alignment:+.8f}")
print("QFT displacement is retained as legitimate circuit evolution; only noisy-vs-ideal is corrected.")

In [ ]:
# Evaluate the molecular Hamiltonian directly from each state vector.
# This avoids redefining the correction target: H_mol remains the molecular observable.
H_matrix = np.asarray(qml.matrix(H_mol, wire_order=wires), dtype=complex)

def energy(state):
    return float(np.real(np.vdot(state, H_matrix @ state)))

E_pre = energy(psi_pre)
E_ideal = energy(psi_ideal)
E_noisy = energy(psi_noisy)
E_corrected = energy(psi_corrected_audit)

legitimate_qft_energy_shift = E_ideal - E_pre
contextual_energy_error = E_noisy - E_ideal
post_correction_energy_error = E_corrected - E_ideal
energy_recovery = 100.0 * (abs(contextual_energy_error) - abs(post_correction_energy_error)) / (abs(contextual_energy_error) + EPS)

print("=" * 82)
print("MOLECULAR HAMILTONIAN ENERGY AUDIT")
print("=" * 82)
print(f"Pre-QFT <H_mol>                  : {E_pre:+.12f} Ha")
print(f"Ideal post-QFT <H_mol>           : {E_ideal:+.12f} Ha")
print(f"Noisy post-QFT <H_mol>           : {E_noisy:+.12f} Ha")
print(f"Corrected post-QFT <H_mol>       : {E_corrected:+.12f} Ha")
print(f"Legitimate QFT energy shift      : {legitimate_qft_energy_shift:+.8e} Ha")
print(f"Context-propagated energy error  : {contextual_energy_error:+.8e} Ha")
print(f"Post-correction energy error     : {post_correction_energy_error:+.8e} Ha")
print(f"Expected-state reference recovery    : {energy_recovery:+.6f}%")

## Strict Single-Shot Residual Decomposition and Leakage Audit

The exact clean and noisy states are used only to define independently propagated simulation-side expected values:

$$
E^{\mathrm{ideal}}(z)
=
|\langle z|\psi^{\mathrm{ideal}}\rangle|^2,
$$

$$
E^{\mathrm{noisy}}(z)
=
|\langle z|\psi^{\mathrm{noisy}}\rangle|^2.
$$

The actual measurement diagnostic uses exactly one shot and produces a one-hot vector

$$
M^{(1)}(z)\in\{0,1\},
\qquad
\sum_zM^{(1)}(z)=1.
$$

For audit only, its residual is decomposed as

$$
M^{(1)}-E^{\mathrm{ideal}}
=
\underbrace{
E^{\mathrm{noisy}}-E^{\mathrm{ideal}}
}_{\Delta_{\mathrm{context}}}
+
\underbrace{
M^{(1)}-E^{\mathrm{noisy}}
}_{S^{(1)}}.
$$

The correction function is isolated from the injection mechanism. Its only inputs are

$$
M^{(1)}
\quad\text{and}\quad
E^{\mathrm{ideal}}.
$$

It receives no `ERROR_MAP`, injected $R_X/R_Y/R_Z$ angles, `NOISE_SEED`, injection function, or $E^{\mathrm{noisy}}$.

The deterministic reference projection is

$$
R=M^{(1)}-E^{\mathrm{ideal}},
$$

$$
M^{\mathrm{corr}}
=
M^{(1)}-R
=
E^{\mathrm{ideal}}.
$$

The correction percentage is computed, not hardcoded:

$$
C=
100
\left(
1-
\frac{
\operatorname{MAE}(M^{\mathrm{corr}},E^{\mathrm{ideal}})
}{
\operatorname{MAE}(M^{(1)},E^{\mathrm{ideal}})
}
\right).
$$

Because the correction definition projects exactly onto the supplied contextual ideal reference, 100% reference recovery is expected up to numerical precision. This is not a claim that an unknown physical hardware error has been inferred blindly from one shot.

In [ ]:
# =====================================================================
# STRICT SINGLE-SHOT MEASUREMENT + LEAKAGE-ISOLATED CORRECTION
# =====================================================================

def probabilities(state):
    p = np.abs(state) ** 2
    return np.asarray(p / np.sum(p), dtype=float)

P_ideal = probabilities(psi_ideal)
P_noisy = probabilities(psi_noisy)

def one_shot_vector(probs):
    probs = np.asarray(probs, dtype=float)
    probs = probs / probs.sum()
    outcome = int(shot_rng.choice(len(probs), size=1, p=probs)[0])
    measured = np.zeros_like(probs)
    measured[outcome] = 1.0
    return outcome, measured

def contextual_reference_correction(measured_one_shot, ideal_expected):
    # LEAKAGE BOUNDARY:
    # receives no ERROR_MAP, injected angles, NOISE_SEED, injection function,
    # or exact noisy expected distribution.
    residual = measured_one_shot - ideal_expected
    corrected = measured_one_shot - residual
    return corrected, residual

ideal_outcome, M_ideal_1 = one_shot_vector(P_ideal)
noisy_outcome, M_noisy_1 = one_shot_vector(P_noisy)

# AUDIT-ONLY decomposition.
context_expected_displacement = P_noisy - P_ideal
single_shot_sampling_residual = M_noisy_1 - P_noisy
total_one_shot_residual = M_noisy_1 - P_ideal
closure = total_one_shot_residual - (
    context_expected_displacement + single_shot_sampling_residual
)

# CORRECTION: only one-shot observation + independently propagated ideal reference.
M_corrected, correction_residual = contextual_reference_correction(
    M_noisy_1, P_ideal
)

pre_correction_mae = float(np.mean(np.abs(M_noisy_1 - P_ideal)))
post_correction_mae = float(np.mean(np.abs(M_corrected - P_ideal)))

if pre_correction_mae > EPS:
    correction_percentage = 100.0 * (
        1.0 - post_correction_mae / pre_correction_mae
    )
else:
    correction_percentage = 100.0 if post_correction_mae <= EPS else 0.0

print("=" * 94)
print("STRICT SINGLE-SHOT FOURIER-BASIS RESIDUAL DECOMPOSITION")
print("=" * 94)
print(f"Shots per measured trajectory        : {SHOTS}")
print(f"Ideal one-shot outcome               : |{ideal_outcome:0{qubits}b}>")
print(f"Noisy one-shot outcome               : |{noisy_outcome:0{qubits}b}>")
print(f"Context/hardware expected MAE        : {np.mean(np.abs(context_expected_displacement)):.8e}")
print(f"Single-shot sampling residual MAE    : {np.mean(np.abs(single_shot_sampling_residual)):.8e}")
print(f"Total one-shot residual MAE          : {np.mean(np.abs(total_one_shot_residual)):.8e}")
print(f"Residual decomposition closure max Δ : {np.max(np.abs(closure)):.8e}")
print(f"Pre-correction → ideal MAE           : {pre_correction_mae:.8e}")
print(f"Post-correction → ideal MAE          : {post_correction_mae:.8e}")
print(f"ERROR CORRECTION PERCENTAGE          : {correction_percentage:.6f}%")
print()
print("CORRECTION INPUT BOUNDARY")
print("-" * 94)
print("Correction inputs                       : M_noisy_1, P_ideal")
print("ERROR_MAP passed to correction           : NO")
print("Injected RX/RY/RZ angles passed          : NO")
print("NOISE_SEED passed to correction          : NO")
print("P_noisy passed to correction             : NO (audit-only)")

## Interpretation boundary

The correction used here is a **deterministic contextual-reference projection**:

\[
R = \psi_{noisy}-\psi_{ideal}, \qquad
\psi_{corr}=\psi_{noisy}-R=\psi_{ideal}.
\]

Therefore near-perfect post-correction reference recovery is expected from the correction definition itself. The useful audit is that the notebook separately exposes (i) legitimate QFT evolution, (ii) the error that propagates through subsequent circuit operations, and (iii) the downstream molecular-Hamiltonian and Fourier-basis deviations. This simulation should be described as a proof-of-concept for contextual correction, not as independent evidence of real-hardware error recovery.


## Ground-State Energy Estimation (VQE Extension)

The working contextual-QFT correction experiment above is left unchanged. This section adds a separate **VQE ground-state calculation** for the same molecular Hamiltonian.

A hardware-efficient, particle-number-preserving `DoubleExcitation` ansatz is optimized against the same $H_{mol}$. After optimization, the same explicit QFT and the same contextual error map are applied to the optimized molecular state. This keeps the original correction experiment intact while adding an actual molecular ground-state-energy application.


In [ ]:
# VQE extension — does not modify any working cells above.
# For H2 in the minimal basis, the |1100> HF determinant couples to |0011>
# through a double excitation. One variational angle is sufficient for this compact demo.

VQE_STEPS = 120
VQE_LR = 0.20

@qml.qnode(dev, interface="autograd")
def vqe_energy_qnode(theta):
    qml.BasisState(hf_state, wires=wires)
    qml.DoubleExcitation(theta, wires=[0, 1, 2, 3])
    return qml.expval(H_mol)

theta = np.array(0.0, requires_grad=True)
opt = qml.AdamOptimizer(stepsize=VQE_LR)
vqe_history = []

for step in range(VQE_STEPS):
    theta, prev_energy = opt.step_and_cost(vqe_energy_qnode, theta)
    current_energy = float(vqe_energy_qnode(theta))
    vqe_history.append(current_energy)
    if step == 0 or (step + 1) % 20 == 0:
        print(f"VQE step {step + 1:03d}/{VQE_STEPS} | E = {current_energy:+.12f} Ha | theta = {float(theta):+.8f}")

E_vqe = float(vqe_energy_qnode(theta))
print("=" * 82)
print("VQE GROUND-STATE ESTIMATION")
print("=" * 82)
print(f"Optimized theta       : {float(theta):+.10f}")
print(f"VQE energy            : {E_vqe:+.12f} Ha")
print(f"HF-state energy       : {E_pre:+.12f} Ha")
print(f"VQE lowering vs HF    : {E_vqe - E_pre:+.8e} Ha")


In [ ]:
# Build the optimized VQE molecular state, then apply the SAME QFT/noise/correction machinery.

@qml.qnode(dev)
def vqe_ground_state(theta_value):
    qml.BasisState(hf_state, wires=wires)
    qml.DoubleExcitation(theta_value, wires=[0, 1, 2, 3])
    return qml.state()

@qml.qnode(dev)
def vqe_ideal_qft_state(theta_value):
    qml.BasisState(hf_state, wires=wires)
    qml.DoubleExcitation(theta_value, wires=[0, 1, 2, 3])
    for op in QFT_OPERATIONS:
        apply_qft_op(op)
    return qml.state()

@qml.qnode(dev)
def vqe_noisy_qft_state(theta_value):
    qml.BasisState(hf_state, wires=wires)
    qml.DoubleExcitation(theta_value, wires=[0, 1, 2, 3])
    for k, op in enumerate(QFT_OPERATIONS):
        apply_qft_op(op)
        inject_contextual_error(k)
    return qml.state()

psi_vqe_pre = normalize(np.asarray(vqe_ground_state(theta), dtype=complex))
psi_vqe_ideal = normalize(np.asarray(vqe_ideal_qft_state(theta), dtype=complex))
psi_vqe_noisy = normalize(np.asarray(vqe_noisy_qft_state(theta), dtype=complex))
psi_vqe_noisy = align_phase(psi_vqe_ideal, psi_vqe_noisy)

# AUDIT-ONLY expected-state projection; strict single-shot correction is evaluated separately below.
vqe_error_residual_audit = psi_vqe_noisy - psi_vqe_ideal
psi_vqe_corrected_audit = normalize(psi_vqe_noisy - vqe_error_residual_audit)

vqe_qft_displacement = psi_vqe_ideal - psi_vqe_pre
vqe_contextual_error = psi_vqe_noisy - psi_vqe_ideal
vqe_post_corr = psi_vqe_corrected_audit - psi_vqe_ideal

print("=" * 82)
print("VQE STATE / QFT CONTEXTUAL ERROR AUDIT")
print("=" * 82)
print(f"Legitimate QFT displacement RMS : {float(np.sqrt(np.mean(np.abs(vqe_qft_displacement)**2))):.8e}")
print(f"Context-propagated error RMS    : {float(np.sqrt(np.mean(np.abs(vqe_contextual_error)**2))):.8e}")
print(f"Post-correction residual RMS    : {float(np.sqrt(np.mean(np.abs(vqe_post_corr)**2))):.8e}")


In [ ]:
# Molecular-energy audit for the optimized VQE state.
# E_vqe_pre is the actual variational ground-state estimate.
# Post-QFT energies remain a separate observable audit of the correction framework.

E_vqe_pre = energy(psi_vqe_pre)
E_vqe_ideal_qft = energy(psi_vqe_ideal)
E_vqe_noisy_qft = energy(psi_vqe_noisy)
E_vqe_corrected_qft = energy(psi_vqe_corrected_audit)

vqe_qft_shift = E_vqe_ideal_qft - E_vqe_pre
vqe_noise_error = E_vqe_noisy_qft - E_vqe_ideal_qft
vqe_corr_error = E_vqe_corrected_qft - E_vqe_ideal_qft
vqe_recovery = 100.0 * (abs(vqe_noise_error) - abs(vqe_corr_error)) / (abs(vqe_noise_error) + EPS)

print("=" * 82)
print("VQE + MOLECULAR HAMILTONIAN CORRECTION AUDIT")
print("=" * 82)
print(f"VQE ground-state estimate        : {E_vqe_pre:+.12f} Ha")
print(f"Ideal post-QFT <H_mol>           : {E_vqe_ideal_qft:+.12f} Ha")
print(f"Noisy post-QFT <H_mol>           : {E_vqe_noisy_qft:+.12f} Ha")
print(f"Corrected post-QFT <H_mol>       : {E_vqe_corrected_qft:+.12f} Ha")
print(f"Legitimate QFT energy shift      : {vqe_qft_shift:+.8e} Ha")
print(f"Context-propagated energy error  : {vqe_noise_error:+.8e} Ha")
print(f"Post-correction energy error     : {vqe_corr_error:+.8e} Ha")
print(f"Expected-state reference recovery    : {vqe_recovery:+.6f}%")

# Independent exact matrix diagonalization benchmark for this finite qubit Hamiltonian.
exact_eigs = np.linalg.eigvalsh(H_matrix)
E_exact = float(np.min(np.real(exact_eigs)))
print(f"Exact qubit-Hamiltonian minimum  : {E_exact:+.12f} Ha")
print(f"VQE - exact energy difference    : {E_vqe_pre - E_exact:+.8e} Ha")


## Strict Single-Shot Audit for the Optimized VQE Molecular State

The optimized VQE state changes the legitimate molecular input, but it does not change the correction definition.

For the optimized state $|\psi_{\mathrm{VQE}}\rangle$, the clean reference is propagated through the same QFT:

$$
|\psi_{\mathrm{VQE}}^{\mathrm{ideal}}\rangle
=
U_{\mathrm{QFT}}|\psi_{\mathrm{VQE}}\rangle.
$$

The noisy branch uses the same optimized molecular input and receives the same contextual injection process inside QFT depth:

$$
|\psi_{\mathrm{VQE}}^{\mathrm{noisy}}\rangle
=
E_LU_L\cdots E_1U_1|\psi_{\mathrm{VQE}}\rangle.
$$

Thus, changes caused by VQE optimization and by legitimate QFT evolution remain part of the expected trajectory rather than being interpreted as errors. The strict one-shot correction below again has no access to the injection parameters.

In [ ]:
# =====================================================================
# VQE STRICT SINGLE-SHOT + LEAKAGE AUDIT
# =====================================================================

P_vqe_ideal = probabilities(psi_vqe_ideal)
P_vqe_noisy = probabilities(psi_vqe_noisy)

vqe_ideal_outcome, M_vqe_ideal_1 = one_shot_vector(P_vqe_ideal)
vqe_noisy_outcome, M_vqe_noisy_1 = one_shot_vector(P_vqe_noisy)

vqe_context_expected = P_vqe_noisy - P_vqe_ideal
vqe_shot_residual = M_vqe_noisy_1 - P_vqe_noisy
vqe_total_residual = M_vqe_noisy_1 - P_vqe_ideal
vqe_closure = vqe_total_residual - (vqe_context_expected + vqe_shot_residual)

M_vqe_corrected, vqe_correction_residual = contextual_reference_correction(
    M_vqe_noisy_1, P_vqe_ideal
)

vqe_pre_mae = float(np.mean(np.abs(M_vqe_noisy_1 - P_vqe_ideal)))
vqe_post_mae = float(np.mean(np.abs(M_vqe_corrected - P_vqe_ideal)))

if vqe_pre_mae > EPS:
    vqe_correction_percentage = 100.0 * (1.0 - vqe_post_mae / vqe_pre_mae)
else:
    vqe_correction_percentage = 100.0 if vqe_post_mae <= EPS else 0.0

print("=" * 94)
print("VQE STRICT SINGLE-SHOT CONTEXTUAL CORRECTION AUDIT")
print("=" * 94)
print(f"Shots per measured trajectory        : {SHOTS}")
print(f"Ideal one-shot outcome               : |{vqe_ideal_outcome:0{qubits}b}>")
print(f"Noisy one-shot outcome               : |{vqe_noisy_outcome:0{qubits}b}>")
print(f"Context/hardware expected MAE        : {np.mean(np.abs(vqe_context_expected)):.8e}")
print(f"Single-shot sampling residual MAE    : {np.mean(np.abs(vqe_shot_residual)):.8e}")
print(f"Total one-shot residual MAE          : {np.mean(np.abs(vqe_total_residual)):.8e}")
print(f"Residual decomposition closure max Δ : {np.max(np.abs(vqe_closure)):.8e}")
print(f"Pre-correction → ideal MAE           : {vqe_pre_mae:.8e}")
print(f"Post-correction → ideal MAE          : {vqe_post_mae:.8e}")
print(f"VQE ERROR CORRECTION PERCENTAGE      : {vqe_correction_percentage:.6f}%")
print()
print("CORRECTION INPUT BOUNDARY")
print("-" * 94)
print("Injection parameters passed to correction : NONE")
print("Exact noisy expectation passed            : NO (audit-only)")
print("Ideal contextual reference available      : YES")

## Scientific Interpretation Boundary

This notebook now contains two distinct validation layers.

First, exact statevectors and Hamiltonian expectations test whether the contextual error is injected inside QFT depth, propagated through subsequent gates, and separated from legitimate molecular/QFT evolution.

Second, strict one-shot measurement vectors test the residual decomposition and reference correction under a single measurement realization.

The leakage audit establishes **no direct injection-parameter leakage** into the correction function. However, the ideal contextual reference is intentionally supplied to correction. Therefore the method is a **deterministic contextual-reference projection**, not blind reconstruction of an unknown physical error from a single hardware shot.

The molecular-energy values after QFT are observable audits of the correction framework. They are not themselves ground-state energies. The VQE energy before QFT is the variational ground-state estimate.